In [15]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the datasets
final_df = pd.read_csv('/content/final_combined.csv', index_col=0)
types_df = pd.read_excel('/content/Formatted_Full_playerinfo.com (2).xlsx')


In [9]:

# Merge the data to enrich final_df with player information
batting_type = types_df[['Player Name', 'Batting type']].set_index('Player Name')
bowling_type = types_df[['Player Name', 'Bowling type']].set_index('Player Name')
playing_role = types_df[['Player Name', 'Playing role']].set_index('Player Name')

final_df = final_df.merge(batting_type, how='left', left_on='Batsman', right_index=True)
final_df = final_df.merge(bowling_type, how='left', left_on='Bowler', right_index=True)
final_df = final_df.merge(playing_role, how='left', left_on='Batsman', right_index=True, suffixes=('_Batting', '_Bowling'))

# Rename columns for clarity
final_df.rename(columns={
    'Batting type': 'Batsman_Batting_type',
    'Bowling type': 'Bowler_Bowling_type',
    'Playing role': 'Batsman_Playing_role'
}, inplace=True)

# Function to train and predict using Random Forest for each metric
def train_and_predict_rf(X_train, y_train, X_test):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return model.predict(X_test)

# Example teams (input for the pipeline)
team_a_batsmen = final_df['Batsman'].unique().tolist()  # Extract unique batsmen names
team_b_bowlers = final_df['Bowler'].unique().tolist()  # Extract unique bowlers names

# Filter the dataset for team matchups
team_a_vs_team_b = final_df[(final_df['Batsman'].isin(team_a_batsmen)) &
                            (final_df['Bowler'].isin(team_b_bowlers))]

# Extract features for Batsman vs Bowler matchups
batsman_vs_bowler_stats = team_a_vs_team_b.groupby(['Batsman', 'Bowler']).agg(
    total_runs=('Runs_off_Bat', 'sum'),
    total_balls_faced=('Over.Ball', 'count'),
    dismissals=('Dissmissed_Player_When_Wicket', 'count')
).reset_index()

batsman_vs_bowler_stats['average'] = batsman_vs_bowler_stats['total_runs'] / batsman_vs_bowler_stats['dismissals'].replace(0, 1)
batsman_vs_bowler_stats['strike_rate'] = (batsman_vs_bowler_stats['total_runs'] / batsman_vs_bowler_stats['total_balls_faced']) * 100

# Extract features for Bowler vs Batsman matchups
bowler_vs_batsman_stats = team_a_vs_team_b.groupby(['Bowler', 'Batsman']).agg(
    total_wickets=('Dissmissed_Player_When_Wicket', 'count'),
    total_runs_conceded=('Runs_off_Bat', 'sum'),
    total_balls_bowled=('Over.Ball', 'count')
).reset_index()

bowler_vs_batsman_stats['bowling_average'] = bowler_vs_batsman_stats['total_runs_conceded'] / bowler_vs_batsman_stats['total_wickets'].replace(0, 1)
bowler_vs_batsman_stats['economy_rate'] = (bowler_vs_batsman_stats['total_runs_conceded'] / bowler_vs_batsman_stats['total_balls_bowled']) * 6

# Split features and targets for the models
X_batsman_vs_bowler = batsman_vs_bowler_stats[['total_balls_faced', 'dismissals']]
y_runs = batsman_vs_bowler_stats['total_runs']
y_strike_rate = batsman_vs_bowler_stats['strike_rate']
y_average = batsman_vs_bowler_stats['average']

X_bowler_vs_batsman = bowler_vs_batsman_stats[['total_balls_bowled', 'total_wickets']]
y_economy_rate = bowler_vs_batsman_stats['economy_rate']
y_wickets = bowler_vs_batsman_stats['total_wickets']
y_bowling_average = bowler_vs_batsman_stats['bowling_average']

# Train and predict for batsman metrics
predicted_runs = train_and_predict_rf(X_batsman_vs_bowler, y_runs, X_batsman_vs_bowler)
predicted_strike_rate = train_and_predict_rf(X_batsman_vs_bowler, y_strike_rate, X_batsman_vs_bowler)
predicted_average = train_and_predict_rf(X_batsman_vs_bowler, y_average, X_batsman_vs_bowler)

# Train and predict for bowler metrics
predicted_economy_rate = train_and_predict_rf(X_bowler_vs_batsman, y_economy_rate, X_bowler_vs_batsman)
predicted_wickets = train_and_predict_rf(X_bowler_vs_batsman, y_wickets, X_bowler_vs_batsman)
predicted_bowling_average = train_and_predict_rf(X_bowler_vs_batsman, y_bowling_average, X_bowler_vs_batsman)

# Add predictions to the stats tables
batsman_vs_bowler_stats['predicted_runs'] = predicted_runs
batsman_vs_bowler_stats['predicted_strike_rate'] = predicted_strike_rate
batsman_vs_bowler_stats['predicted_average'] = predicted_average

bowler_vs_batsman_stats['predicted_economy_rate'] = predicted_economy_rate
bowler_vs_batsman_stats['predicted_wickets'] = predicted_wickets
bowler_vs_batsman_stats['predicted_bowling_average'] = predicted_bowling_average

# Evaluate the model's performance
def assess_accuracy(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, r2

# Calculate accuracy for each prediction
batsman_accuracy = {
    "Runs": assess_accuracy(y_runs, predicted_runs),
    "Strike Rate": assess_accuracy(y_strike_rate, predicted_strike_rate),
    "Average": assess_accuracy(y_average, predicted_average)
}

bowler_accuracy = {
    "Economy Rate": assess_accuracy(y_economy_rate, predicted_economy_rate),
    "Wickets": assess_accuracy(y_wickets, predicted_wickets),
    "Bowling Average": assess_accuracy(y_bowling_average, predicted_bowling_average)
}

# Display the results
print("Batsman vs Bowler Predictions:")
print(batsman_vs_bowler_stats)

print("\nBowler vs Batsman Predictions:")
print(bowler_vs_batsman_stats)

print("\nBatsman Accuracy (MAE, MSE, R²):")
print(batsman_accuracy)

print("\nBowler Accuracy (MAE, MSE, R²):")
print(bowler_accuracy)

Batsman vs Bowler Predictions:
                 Batsman        Bowler  total_runs  total_balls_faced  \
0                A Bosch      A Capsey           5                  3   
1                A Bosch     A Gardner           7                 10   
2                A Bosch       A Reddy          18                 16   
3                A Bosch  A Sutherland           9                  7   
4                A Bosch  AC Jayangani          16                 13   
...                  ...           ...         ...                ...   
43006  Zulqarnain Haider       J Botha           1                  1   
43007  Zulqarnain Haider      J Theron          10                  6   
43008  Zulqarnain Haider     JA Morkel           4                  4   
43009  Zulqarnain Haider   LL Tsotsobe           1                  5   
43010  Zulqarnain Haider      M Morkel           2                  6   

       dismissals  average  strike_rate  predicted_runs  \
0               0      5.0   166.

In [4]:
# Rename columns for clarity
final_df.rename(columns={
    'Batting type': 'Batsman_Batting_type',
    'Bowling type': 'Bowler_Bowling_type',
    'Playing role': 'Batsman_Playing_role'
}, inplace=True)


In [12]:

# Display the results
print("Batsman vs Bowler Predictions:")
print(batsman_vs_bowler_stats.to_csv('batsman_vs_bowler_stats.csv'))

print("\nBowler vs Batsman Predictions:")
print(bowler_vs_batsman_stats.to_csv('bowler_vs_batsman_stats.csv'))

print("\nBatsman Accuracy (MAE, MSE, R²):")
print(pd.DataFrame(batsman_accuracy).to_csv('batsman_accuracy.csv'))

print("\nBowler Accuracy (MAE, MSE, R²):")
print(pd.DataFrame(bowler_accuracy).to_csv('bowler_accuracy.csv'))

Batsman vs Bowler Predictions:
None

Bowler vs Batsman Predictions:
None

Batsman Accuracy (MAE, MSE, R²):
None

Bowler Accuracy (MAE, MSE, R²):
None


In [6]:
final_df

,Inning_Number,Over.Ball,Batting_Team,Batsman,Non_Stricker,Bowler,Runs_off_Bat,Extras,Wides,No_Balls,...,date,toss_winner,toss_decision,venue,match_id,Batsman_Batting_type,Bowler_Bowling_type,Batsman_Playing_role,balls_faced,total_runs
0,1,0.1,Zimbabwe,CJ Chibhabha,H Masakadza,JD Unadkat,0,0,NaN,NaN,...,2016-06-18,India,field,Harare Sports Club,t20s_csv/1007655.csv,NaN,Left arm Medium,NaN,1,0
1,1,0.2,Zimbabwe,CJ Chibhabha,H Masakadza,JD Unadkat,0,0,NaN,NaN,...,2016-06-18,India,field,Harare Sports Club,t20s_csv/1007655.csv,NaN,Left arm Medium,NaN,2,0
2,1,0.3,Zimbabwe,CJ Chibhabha,H Masakadza,JD Unadkat,0,0,NaN,NaN,...,2016-06-18,India,field,Harare Sports Club,t20s_csv/1007655.csv,NaN,Left arm Medium,NaN,3,0
3,1,0.4,Zimbabwe,CJ Chibhabha,H Masakadza,JD Unadkat,1,0,NaN,NaN,...,2016-06-18,India,field,Harare Sports Club,t20s_csv/1007655.csv,NaN,Left arm Medium,NaN,4,1
4,1,0.5,Zimbabwe,H Masakadza,CJ Chibhabha,JD Unadkat,0,0,NaN,NaN,...,2016-06-18,India,field,Harare Sports Club,t20s_csv/1007655.csv,NaN,Left arm Medium,NaN,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,2,14.5,England,JM Bairstow,HC Brook,Naseem Shah,0,0,NaN,NaN,...,2024-05-30,England,field,"Kennington Oval, London",t20s_csv/1385690.csv,Right hand Bat,Right arm Fast,Wicketkeeper Batter,980,1374
215,2,14.6,England,JM Bairstow,HC Brook,Naseem Shah,0,0,NaN,NaN,...,2024-05-30,England,field,"Kennington Oval, London",t20s_csv/1385690.csv,Right hand Bat,Right arm Fast,Wicketkeeper Batter,981,1374
216,2,15.1,England,HC Brook,JM Bairstow,Haris Rauf,4,0,NaN,NaN,...,2024-05-30,England,field,"Kennington Oval, London",t20s_csv/1385690.csv,Right hand Bat,Right arm Fast,Batter,401,556
217,2,15.2,England,HC Brook,JM Bairstow,Haris Rauf,0,0,NaN,NaN,...,2024-05-30,England,field,"Kennington Oval, London",t20s_csv/1385690.csv,Right hand Bat,Right arm Fast,Batter,402,556


In [7]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 337629 entries, 0 to 242
Data columns (total 31 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Inning_Number                  337629 non-null  int64         
 1   Over.Ball                      337629 non-null  float64       
 2   Batting_Team                   337629 non-null  object        
 3   Batsman                        337629 non-null  object        
 4   Non_Stricker                   337629 non-null  object        
 5   Bowler                         337629 non-null  object        
 6   Runs_off_Bat                   337629 non-null  int64         
 7   Extras                         337629 non-null  int64         
 8   Wides                          10782 non-null   float64       
 9   No_Balls                       1333 non-null    float64       
 10  Byes                           1054 non-null    float64       
 11  Leg_Byes

----


In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the additional player info dataset
player_info = pd.read_excel('/content/Formatted_Full_playerinfo.com (2).xlsx')

# Prepare the player information data
batting_type = player_info[['Player Name', 'Batting type']].set_index('Player Name')
bowling_type = player_info[['Player Name', 'Bowling type']].set_index('Player Name')
playing_role = player_info[['Player Name', 'Playing role']].set_index('Player Name')

# Function to process each chunk
def process_chunk(chunk):
    # Convert date column to datetime format
    chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')

    # Handle missing values in a straightforward way
    chunk.fillna({
        'Wides': 0,
        'No_Balls': 0,
        'Byes': 0,
        'Leg_Byes': 0,
        'Kind_of_wicket_If_Any': 'Not Out',
        'Dissmissed_Player_When_Wicket': 'Not Out',
    }, inplace=True)

    # Merge with player info data
    chunk = chunk.merge(batting_type, how='left', left_on='Batsman', right_index=True)
    chunk = chunk.merge(bowling_type, how='left', left_on='Bowler', right_index=True)
    chunk = chunk.merge(playing_role, how='left', left_on='Batsman', right_index=True)

    # Calculate simple cumulative sums for runs and balls faced
    chunk['balls_faced'] = chunk.groupby(['Batsman']).cumcount() + 1
    chunk['total_runs'] = chunk.groupby(['Batsman'])['Runs_off_Bat'].cumsum()

    # Calculate dismissals
    chunk['dismissals'] = (chunk['Dissmissed_Player_When_Wicket'] != 'Not Out').astype(int)
    chunk['cumulative_dismissals'] = chunk.groupby('Batsman')['dismissals'].cumsum()

    # Calculate current Strike Rate and Average
    chunk['current_strike_rate'] = (chunk['total_runs'] / chunk['balls_faced']) * 100
    chunk['current_average'] = chunk['total_runs'] / chunk['cumulative_dismissals'].replace(0, pd.NA)

    # Calculate cumulative sums for bowlers
    chunk['balls_bowled'] = chunk.groupby(['Bowler']).cumcount() + 1
    chunk['total_runs_conceded'] = chunk.groupby(['Bowler'])['Runs_off_Bat'].cumsum()
    chunk['total_wickets'] = chunk.groupby(['Bowler'])['dismissals'].cumsum()

    # Calculate current Economy Rate and Bowling Average
    chunk['current_economy_rate'] = (chunk['total_runs_conceded'] / chunk['balls_bowled']) * 6
    chunk['current_bowling_average'] = chunk['total_runs_conceded'] / chunk['total_wickets'].replace(0, pd.NA)

    return chunk

# Initialize an empty DataFrame to store the processed data
processed_df = pd.DataFrame()

# Load the dataset in chunks and process each chunk
chunk_size = 10000  # Define the chunk size
chunks = pd.read_csv('/content/final_combined.csv', chunksize=chunk_size)

for chunk in chunks:
    processed_chunk = process_chunk(chunk)
    processed_df = pd.concat([processed_df, processed_chunk], ignore_index=True)

# After processing, perform additional feature engineering and prepare for model training
# Create target variables for the next match performance
processed_df['next_match_runs'] = processed_df.groupby('Batsman')['total_runs'].shift(-1)
processed_df['next_match_strike_rate'] = processed_df.groupby('Batsman')['current_strike_rate'].shift(-1)
processed_df['next_match_average'] = processed_df.groupby('Batsman')['current_average'].shift(-1)

processed_df['next_match_economy_rate'] = processed_df.groupby('Bowler')['current_economy_rate'].shift(-1)
processed_df['next_match_average_bowling'] = processed_df.groupby('Bowler')['current_bowling_average'].shift(-1)
processed_df['next_match_wickets'] = processed_df.groupby('Bowler')['total_wickets'].shift(-1)

# Drop rows where the target variables are NaN (i.e., no next match data)
processed_df.dropna(subset=[
    'next_match_runs', 'next_match_strike_rate', 'next_match_average',
    'next_match_economy_rate', 'next_match_average_bowling', 'next_match_wickets'
], inplace=True)

# Feature Selection for Batsman
X_batsman = processed_df[['current_strike_rate', 'current_average']]
y_runs = processed_df['next_match_runs']
y_strike_rate = processed_df['next_match_strike_rate']
y_average = processed_df['next_match_average']

# Feature Selection for Bowler
X_bowler = processed_df[['current_economy_rate', 'current_bowling_average']]
y_economy_rate = processed_df['next_match_economy_rate']
y_wickets = processed_df['next_match_wickets']
y_bowling_average = processed_df['next_match_average_bowling']

# Split the data into training and testing sets for both batsman and bowler models
X_train_batsman, X_test_batsman, y_train_runs, y_test_runs = train_test_split(X_batsman, y_runs, test_size=0.2, random_state=42)
_, _, y_train_strike_rate, y_test_strike_rate = train_test_split(X_batsman, y_strike_rate, test_size=0.2, random_state=42)
_, _, y_train_average, y_test_average = train_test_split(X_batsman, y_average, test_size=0.2, random_state=42)

X_train_bowler, X_test_bowler, y_train_economy_rate, y_test_economy_rate = train_test_split(X_bowler, y_economy_rate, test_size=0.2, random_state=42)
_, _, y_train_wickets, y_test_wickets = train_test_split(X_bowler, y_wickets, test_size=0.2, random_state=42)
_, _, y_train_bowling_average, y_test_bowling_average = train_test_split(X_bowler, y_bowling_average, test_size=0.2, random_state=42)

# Initialize and train Random Forest models for each target variable
model_runs = RandomForestRegressor(n_estimators=100, random_state=42)
model_strike_rate = RandomForestRegressor(n_estimators=100, random_state=42)
model_average = RandomForestRegressor(n_estimators=100, random_state=42)

model_economy_rate = RandomForestRegressor(n_estimators=100, random_state=42)
model_wickets = RandomForestRegressor(n_estimators=100, random_state=42)
model_bowling_average = RandomForestRegressor(n_estimators=100, random_state=42)
import numpy as np


# Replace pd.NA with np.nan
X_train_batsman.replace({pd.NA: np.nan}, inplace=True)
X_test_batsman.replace({pd.NA: np.nan}, inplace=True)

X_train_bowler.replace({pd.NA: np.nan}, inplace=True)
X_test_bowler.replace({pd.NA: np.nan}, inplace=True)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train_batsman_imputed = imputer.fit_transform(X_train_batsman)
X_test_batsman_imputed = imputer.transform(X_test_batsman)

X_train_bowler_imputed = imputer.fit_transform(X_train_bowler)
X_test_bowler_imputed = imputer.transform(X_test_bowler)

# Train the models with the imputed data
model_runs.fit(X_train_batsman_imputed, y_train_runs)
model_strike_rate.fit(X_train_batsman_imputed, y_train_strike_rate)
model_average.fit(X_train_batsman_imputed, y_train_average)

model_economy_rate.fit(X_train_bowler_imputed, y_train_economy_rate)
model_wickets.fit(X_train_bowler_imputed, y_train_wickets)
model_bowling_average.fit(X_train_bowler_imputed, y_train_bowling_average)

# Predict on the test set
y_pred_runs = model_runs.predict(X_test_batsman_imputed)
y_pred_strike_rate = model_strike_rate.predict(X_test_batsman_imputed)
y_pred_average = model_average.predict(X_test_batsman_imputed)

y_pred_economy_rate = model_economy_rate.predict(X_test_bowler_imputed)
y_pred_wickets = model_wickets.predict(X_test_bowler_imputed)
y_pred_bowling_average = model_bowling_average.predict(X_test_bowler_imputed)

# Evaluate the models for batsman performance
mae_runs = mean_absolute_error(y_test_runs, y_pred_runs)
mse_runs = mean_squared_error(y_test_runs, y_pred_runs)
r2_runs = r2_score(y_test_runs, y_pred_runs)

mae_strike_rate = mean_absolute_error(y_test_strike_rate, y_pred_strike_rate)
mse_strike_rate = mean_squared_error(y_test_strike_rate, y_pred_strike_rate)
r2_strike_rate = r2_score(y_test_strike_rate, y_pred_strike_rate)

mae_average = mean_absolute_error(y_test_average, y_pred_average)
mse_average = mean_squared_error(y_test_average, y_pred_average)
r2_average = r2_score(y_test_average, y_pred_average)

# Compile the evaluation results for batsman
batsman_evaluation = {
    "Metric": ["Runs", "Strike Rate", "Average"],
    "MAE": [mae_runs, mae_strike_rate, mae_average],
    "MSE": [mse_runs, mse_strike_rate, mse_average],
    "R²": [r2_runs, r2_strike_rate, r2_average]
}
batsman_evaluation_df = pd.DataFrame(batsman_evaluation)

# Evaluate the models for bowler performance
mae_economy_rate = mean_absolute_error(y_test_economy_rate, y_pred_economy_rate)
mse_economy_rate = mean_squared_error(y_test_economy_rate, y_pred_economy_rate)
r2_economy_rate = r2_score(y_test_economy_rate, y_pred_economy_rate)

mae_wickets = mean_absolute_error(y_test_wickets, y_pred_wickets)
mse_wickets = mean_squared_error(y_test_wickets, y_pred_wickets)
r2_wickets = r2_score(y_test_wickets, y_pred_wickets)

mae_bowling_average = mean_absolute_error(y_test_bowling_average, y_pred_bowling_average)
mse_bowling_average = mean_squared_error(y_test_bowling_average, y_pred_bowling_average)
r2_bowling_average = r2_score(y_test_bowling_average, y_pred_bowling_average)

# Compile the evaluation results for bowler
bowler_evaluation = {
    "Metric": ["Economy Rate", "Wickets", "Bowling Average"],
    "MAE": [mae_economy_rate, mae_wickets, mae_bowling_average],
    "MSE": [mse_economy_rate, mse_wickets, mse_bowling_average],
    "R²": [r2_economy_rate, r2_wickets, r2_bowling_average]
}
bowler_evaluation_df = pd.DataFrame(bowler_evaluation)

# Display the evaluation metrics for both batsmen and bowlers
# import ace_tools as tools
# tools.display_dataframe_to_user(name="Batsman Model Evaluation", dataframe=batsman_evaluation_df)
# tools.display_dataframe_to_user(name="Bowler Model Evaluation", dataframe=bowler_evaluation_df)

# Output the evaluation DataFrames for final assessment
batsman_evaluation_df, bowler_evaluation_df


(        Metric        MAE         MSE        R²
 0         Runs  11.133809  562.752789  0.833407
 1  Strike Rate   3.615571   68.111343  0.958081
 2      Average   1.887815   27.056149  0.956249,
             Metric       MAE        MSE        R²
 0     Economy Rate  0.182901   0.150283  0.966347
 1          Wickets  0.694345   1.610108  0.786606
 2  Bowling Average  1.250692  11.710587  0.952973)

In [15]:
from google.colab import files
files.download('/content/batsman_vs_bowler_stats.csv')
files.download('/content/bowler_vs_batsman_stats.csv')
files.download('/content/batsman_accuracy.csv')
files.download('/content/bowler_accuracy.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>